In [12]:
from pathlib import Path
import sys
import pandas as pd

# Set project root
# Locate the project folder from wherever the notebook is opened
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "model_setup.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Enable imports from the project
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import training setup
from src.data_preparation import load_training_bundle
from src.model_setup import make_model_pipeline, shared_cv

# Load shared training dataset
X_train, y_train, metadata = load_training_bundle(ROOT)

print("Training Features:", X_train.shape)
print("Training Target:", y_train.shape)
print("Metadata:", metadata.shape)

print("\nTarget Distribution:")
print(y_train.value_counts())

Training Features: (202795, 21)
Training Target: (202795,)
Metadata: (202795, 4)

Target Distribution:
HeartDiseaseorAttack
0    183698
1     19097
Name: count, dtype: int64


In [13]:
from sklearn.tree import DecisionTreeClassifier

# Create Decision Tree baseline model
dt_model = make_model_pipeline(
    DecisionTreeClassifier(
        random_state=42
    ),
    kind="tree"
)

# Use shared 5-fold cross-validation
cv = shared_cv(metadata)

print("Decision Tree Baseline Model Created")

print("\nModel Configuration:")
print(dt_model.named_steps["model"])

print("\nNumber of CV Splits:", cv.get_n_splits())

Decision Tree Baseline Model Created

Model Configuration:
DecisionTreeClassifier(random_state=42)

Number of CV Splits: 5


In [14]:
import pandas as pd
from sklearn.model_selection import cross_validate

# Evaluation metrics
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC_AUC": "roc_auc",
    "Average_Precision": "average_precision"
}

# Train and evaluate using shared 5-fold CV
dt_cv_results = cross_validate(
    estimator=dt_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=True
)

# Summarise validation results
dt_summary = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Mean Validation Score": [
        dt_cv_results[f"test_{metric}"].mean()
        for metric in scoring
    ],
    "Standard Deviation": [
        dt_cv_results[f"test_{metric}"].std()
        for metric in scoring
    ]
})

print("Decision Tree Baseline - 5-Fold Cross-Validation")
display(dt_summary.round(4))

Decision Tree Baseline - 5-Fold Cross-Validation


,Metric,Mean Validation Score,Standard Deviation
0,Accuracy,0.8521,0.0007
1,Precision,0.2459,0.0032
2,Recall,0.2758,0.0059
3,F1,0.2600,0.0044
4,ROC_AUC,0.5962,0.0030
5,Average_Precision,0.1379,0.0022


In [15]:
# Compare training and validation performance
dt_overfitting = pd.DataFrame({
    "Metric": ["Accuracy", "F1", "ROC_AUC", "Average_Precision"],
    "Training Score": [
        dt_cv_results[f"train_{metric}"].mean()
        for metric in ["Accuracy", "F1", "ROC_AUC", "Average_Precision"]
    ],
    "Validation Score": [
        dt_cv_results[f"test_{metric}"].mean()
        for metric in ["Accuracy", "F1", "ROC_AUC", "Average_Precision"]
    ]
})

dt_overfitting["Gap"] = (
    dt_overfitting["Training Score"]
    - dt_overfitting["Validation Score"]
)

print("Decision Tree Training vs Validation Performance")
display(dt_overfitting.round(4))

Decision Tree Training vs Validation Performance


,Metric,Training Score,Validation Score,Gap
0,Accuracy,0.9953,0.8521,0.1432
1,F1,0.9746,0.2600,0.7146
2,ROC_AUC,0.9998,0.5962,0.4037
3,Average_Precision,0.9981,0.1379,0.8601


In [16]:
# Fit Decision Tree on the complete training dataset
dt_model.fit(X_train, y_train)

# Access the trained Decision Tree inside the pipeline
trained_tree = dt_model.named_steps["model"]

# Examine model complexity
print("Decision Tree Model Complexity")
print("Tree Depth:", trained_tree.get_depth())
print("Number of Leaf Nodes:", trained_tree.get_n_leaves())
print("Number of Nodes:", trained_tree.tree_.node_count)

Decision Tree Model Complexity
Tree Depth: 43
Number of Leaf Nodes: 27795
Number of Nodes: 55589


In [17]:
import pandas as pd

# Get feature names after preprocessing
feature_names = dt_model.named_steps["preprocess"].get_feature_names_out()

# Get feature importance from trained Decision Tree
importance_values = trained_tree.feature_importances_

# Create feature importance dataframe
dt_feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})

# Sort features from highest to lowest importance
dt_feature_importance = dt_feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("Decision Tree - Top 10 Important Features")
display(dt_feature_importance.head(10).round(4))

Decision Tree - Top 10 Important Features


,Feature,Importance
0,numeric__BMI,0.1898
19,ordinal__Income,0.1121
17,ordinal__Age,0.0891
2,numeric__PhysHlth,0.0886
16,ordinal__GenHlth,0.0809
18,ordinal__Education,0.0763
1,numeric__MentHlth,0.0658
9,binary__Fruits,0.0366
8,binary__PhysActivity,0.0328
10,binary__Veggies,0.0289


In [18]:
from pathlib import Path
import pandas as pd

# Create results folder
results_dir = ROOT / "results" / "member2"
results_dir.mkdir(parents=True, exist_ok=True)

# Save baseline performance results
dt_summary.to_csv(
    results_dir / "decision_tree_baseline.csv",
    index=False
)

# Save training and validation comparison
dt_overfitting.to_csv(
    results_dir / "decision_tree_overfitting.csv",
    index=False
)

# Save feature importance
dt_feature_importance.to_csv(
    results_dir / "decision_tree_feature_importance.csv",
    index=False
)

# Save tree complexity
dt_complexity = pd.DataFrame([{
    "Tree Depth": trained_tree.get_depth(),
    "Leaf Nodes": trained_tree.get_n_leaves(),
    "Total Nodes": trained_tree.tree_.node_count
}])

dt_complexity.to_csv(
    results_dir / "decision_tree_complexity.csv",
    index=False
)

print("All Decision Tree Baseline Results Saved Successfully")

All Decision Tree Baseline Results Saved Successfully
